# Geocode Cambridge PD quarterly crashes with `human_address`

Run this after `cambridge-police-crash-cleaner.ipynb`. It re-fetches the same Socrata table in date order, finds source `location` objects that have a `human_address` but lack a complete latitude/longitude pair, and fills coordinates in the cleaner's CSV when the Census geocoder finds an in-bounds result. It preserves every input row and writes a separate output file.

The cleaner generates IDs from source row positions. This notebook checks those IDs against source timestamps and street names before applying any geocodes; it stops if the source and cleaned CSV do not align.

## Setup

The Census batch geocoder is used for these street addresses. Results are cached locally so completed lookups are not repeated. The output and cache are kept next to this notebook.

In [5]:
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import sys
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path.cwd()
if not (DATA_DIR / "geocode_cambridge_crashes.py").exists():
    DATA_DIR = DATA_DIR / "data"

SCRIPT_FILE = DATA_DIR / "geocode_cambridge_crashes.py"
INPUT_FILE = DATA_DIR / "cambridge-police-crashes.csv"
OUTPUT_FILE = DATA_DIR / "cambridge-police-crashes-human-address-geocoded.csv"
CACHE_FILE = DATA_DIR / "cambridge-human-address-geocode-cache.json"

for required_file in (SCRIPT_FILE, INPUT_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(f"Required file not found: {required_file}")

if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

from geocode_cambridge_crashes import CENSUS_CHUNK, census_batch, in_bounds

DATA_URL = "https://data.cambridgema.gov/resource/gb5w-yva3.json"
PAGE_SIZE = 5000
REQUEST_TIMEOUT = 60
START_YEAR = 2021

## Fetch and match the source rows

The request uses the cleaner's date ordering and pagination. Before geocoding, the source row positions encoded in `CPD_<row>` IDs are checked against the cleaner output.

In [8]:
def fetch_dataset() -> pd.DataFrame:
    pages = []
    offset = 0

    while True:
        response = requests.get(
            DATA_URL,
            params={
                "$limit": PAGE_SIZE,
                "$offset": offset,
                "$order": "date_time ASC",
            },
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        page = response.json()
        if not isinstance(page, list):
            raise RuntimeError("Socrata returned an unexpected response format")

        pages.extend(page)
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    return pd.DataFrame.from_records(pages)


crash_raw = fetch_dataset()
cambridge_clean = pd.read_csv(INPUT_FILE, dtype={"id": "string"})

required_source_columns = {"date_time", "street_name", "location"}
missing_source_columns = required_source_columns.difference(crash_raw.columns)
if missing_source_columns:
    raise ValueError(f"CPD source is missing columns: {sorted(missing_source_columns)}")
required_clean_columns = {"id", "datetime", "street_name", "lat", "lng"}
missing_clean_columns = required_clean_columns.difference(cambridge_clean.columns)
if missing_clean_columns:
    raise ValueError(f"Cleaner CSV is missing columns: {sorted(missing_clean_columns)}")
if cambridge_clean["id"].isna().any() or not cambridge_clean["id"].is_unique:
    raise ValueError("Cleaner CSV IDs must be present and unique")

crash_raw = crash_raw.reset_index(drop=True)
crash_raw["_source_id"] = "CPD_" + crash_raw.index.astype(str)
crash_raw["_source_datetime"] = pd.to_datetime(
    crash_raw["date_time"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")
source_by_id = crash_raw.set_index("_source_id")
clean_ids = cambridge_clean["id"].astype("string")
missing_source_ids = clean_ids[~clean_ids.isin(source_by_id.index)]
if not missing_source_ids.empty:
    raise ValueError(
        f"{len(missing_source_ids)} cleaner IDs are absent from the current source; "
        "rerun the cleaner notebook against the current quarterly data."
    )

aligned_source = source_by_id.loc[clean_ids.tolist()]
clean_datetime = cambridge_clean["datetime"].fillna("").astype(str).reset_index(drop=True)
source_datetime = aligned_source["_source_datetime"].fillna("").astype(str).reset_index(drop=True)
clean_street = cambridge_clean["street_name"].fillna("").astype(str).str.strip().reset_index(drop=True)
source_street = aligned_source["street_name"].fillna("").astype(str).str.strip().reset_index(drop=True)
datetime_mismatch = clean_datetime.ne(source_datetime)
street_mismatch = clean_street.ne(source_street)

print("Timestamp mismatches:", datetime_mismatch.sum())
print("Street mismatches:", street_mismatch.sum())

display(pd.DataFrame({
    "id": clean_ids,
    "clean_datetime": clean_datetime,
    "source_datetime": source_datetime,
    "clean_street": clean_street,
    "source_street": source_street,
}).loc[datetime_mismatch | street_mismatch].head(20))

print(f"Source records retrieved: {len(crash_raw):,}")
print(f"Cleaner records loaded: {len(cambridge_clean):,}")

Timestamp mismatches: 0
Street mismatches: 2


,id,clean_datetime,source_datetime,clean_street,source_street
8,CPD_8517,2021-01-05 22:40:00,2021-01-05 22:40:00,,N/A
8544,CPD_17053,2026-08-25 10:34:00,2026-08-25 10:34:00,,N/A


Source records retrieved: 17,084
Cleaner records loaded: 8,575


## Identify locations to geocode

Only records from 2021 onward with a `human_address` and at least one missing coordinate are considered. The source address may be a dictionary or a JSON-encoded string.

In [9]:
def location_parts(value):
    if not isinstance(value, dict):
        return False, True, None

    human_address = value.get("human_address")
    if human_address is None or human_address == "":
        return False, True, None

    latitude = pd.to_numeric(value.get("latitude"), errors="coerce")
    longitude = pd.to_numeric(value.get("longitude"), errors="coerce")
    has_coordinates = pd.notna(latitude) and pd.notna(longitude)

    if isinstance(human_address, str):
        try:
            parsed_address = json.loads(human_address)
        except json.JSONDecodeError:
            parsed_address = {"address": human_address}
    else:
        parsed_address = human_address

    if isinstance(parsed_address, str):
        parsed_address = {"address": parsed_address}
    if not isinstance(parsed_address, dict):
        return True, not has_coordinates, None

    address = parsed_address.get("address") or parsed_address.get("street")
    if not isinstance(address, str) or not address.strip():
        return True, not has_coordinates, None
    return True, not has_coordinates, address.strip()


source_location_parts = crash_raw["location"].map(location_parts)
crash_raw[["_has_human_address", "_needs_coordinates", "_address_query"]] = pd.DataFrame(
    source_location_parts.tolist(), index=crash_raw.index
)
source_year = pd.to_datetime(crash_raw["date_time"], errors="coerce").dt.year
target_source = crash_raw.loc[
    crash_raw["_has_human_address"]
    & crash_raw["_needs_coordinates"]
    & source_year.ge(START_YEAR)
].copy()

clean_id_set = set(clean_ids)
target_source = target_source.loc[target_source["_source_id"].isin(clean_id_set)].copy()
target_source["_clean_row"] = target_source["_source_id"].map(
    cambridge_clean.reset_index().set_index("id")["index"]
)
unusable_addresses = target_source["_address_query"].isna()
print(f"Rows with human_address and missing coordinates: {len(target_source):,}")
print(f"Rows with a usable street address: {(~unusable_addresses).sum():,}")
print(f"Rows with an unusable human_address value: {unusable_addresses.sum():,}")
display(target_source[["_source_id", "_address_query"]].head(10))

Rows with human_address and missing coordinates: 3,180
Rows with a usable street address: 3,180
Rows with an unusable human_address value: 0


,_source_id,_address_query
8511,CPD_8511,GARDEN STREET
8513,CPD_8513,PROSPECT STREET
8516,CPD_8516,CAMBRIDGE ST
8517,CPD_8517,0 N A
8518,CPD_8518,SPRING ST
8519,CPD_8519,ALBANY ST
8521,CPD_8521,CAMBRIDGE STREET
8524,CPD_8524,5 LOT
8525,CPD_8525,WINDSOR
8526,CPD_8526,OSBORN ST


## Geocode, apply in-bounds matches, and save

Census requests are sent in batches and successful/no-match results are cached. Results outside the Cambridge bounding box are not written as coordinates.

In [10]:
if CACHE_FILE.exists():
    geocode_cache = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
    if not isinstance(geocode_cache, dict):
        raise ValueError(f"Geocoding cache must contain a JSON object: {CACHE_FILE}")
else:
    geocode_cache = {}

address_by_key = {
    "H|" + address.casefold(): address
    for address in target_source["_address_query"].dropna().unique()
}
uncached_keys = sorted(key for key in address_by_key if key not in geocode_cache)
print(f"Unique usable addresses: {len(address_by_key):,}")
print(f"Addresses not yet cached: {len(uncached_keys):,}")

for start in range(0, len(uncached_keys), CENSUS_CHUNK):
    keys = uncached_keys[start : start + CENSUS_CHUNK]
    queries = [(str(position), address_by_key[key]) for position, key in enumerate(keys)]
    batch_results = census_batch(queries)
    for position, key in enumerate(keys):
        result = batch_results.get(str(position))
        if result is not None and result["status"] != "error":
            geocode_cache[key] = {**result, "source": "census"}
    CACHE_FILE.write_text(json.dumps(geocode_cache), encoding="utf-8")
    print(f"Completed address batch {start // CENSUS_CHUNK + 1}")

for column in ("geocode_source", "geocode_status", "matched_address", "match_quality"):
    if column not in cambridge_clean.columns:
        cambridge_clean[column] = pd.NA
cambridge_clean["geocode_status"] = cambridge_clean["geocode_status"].fillna("not_attempted")

for _, row in target_source.iterrows():
    clean_row = int(row["_clean_row"])
    address = row["_address_query"]
    if pd.isna(address):
        cambridge_clean.loc[clean_row, "geocode_status"] = "invalid_human_address"
        continue

    key = "H|" + address.casefold()
    result = geocode_cache.get(key)
    if result is None:
        cambridge_clean.loc[clean_row, "geocode_status"] = "lookup_failed_rerun"
        continue

    cambridge_clean.loc[clean_row, "geocode_source"] = result.get("source", "census")
    cambridge_clean.loc[clean_row, "matched_address"] = result.get("matched")
    cambridge_clean.loc[clean_row, "match_quality"] = result.get("quality")
    cambridge_clean.loc[clean_row, "geocode_status"] = result["status"]
    if result["status"] == "match":
        latitude = float(result["lat"])
        longitude = float(result["lon"])
        if in_bounds(latitude, longitude):
            cambridge_clean.loc[clean_row, ["lat", "lng"]] = [latitude, longitude]
        else:
            cambridge_clean.loc[clean_row, ["lat", "lng"]] = pd.NA
            cambridge_clean.loc[clean_row, "geocode_status"] = "out_of_bounds"

cambridge_clean.to_csv(OUTPUT_FILE, index=False)
print("\nGeocode status counts:")
print(cambridge_clean["geocode_status"].value_counts(dropna=False).to_string())
print(f"\nSaved geocoded data to {OUTPUT_FILE}")
display(cambridge_clean.loc[
    cambridge_clean["geocode_status"].ne("not_attempted"),
    ["id", "lat", "lng", "geocode_source", "geocode_status", "matched_address"],
].head(10))

Unique usable addresses: 689
Addresses not yet cached: 689
Completed address batch 1

Geocode status counts:
geocode_status
not_attempted    5395
no_match         3162
match              18

Saved geocoded data to c:\Users\johnbest\mass-crash-map\data\cambridge-police-crashes-human-address-geocoded.csv


,id,lat,lng,geocode_source,geocode_status,matched_address
2,CPD_8511,NaN,NaN,census,no_match,None
4,CPD_8513,NaN,NaN,census,no_match,None
7,CPD_8516,NaN,NaN,census,no_match,None
8,CPD_8517,NaN,NaN,census,no_match,None
9,CPD_8518,NaN,NaN,census,no_match,None
10,CPD_8519,NaN,NaN,census,no_match,None
12,CPD_8521,NaN,NaN,census,no_match,None
15,CPD_8524,NaN,NaN,census,no_match,None
16,CPD_8525,NaN,NaN,census,no_match,None
17,CPD_8526,NaN,NaN,census,no_match,None
